# COMP4703 Assignment 1: The Biography Fact Extractor

## How this notebook works

Every cell marked **`# YOUR CODE HERE`** is something you must implement
— the markdown cell above it specifies the exact specification. Run cells
top-to-bottom; later parts depend on earlier ones.

After each exercise there is a **checks cell**. It does *not* use
`assert` (which would stop at the first failure and hide the rest) —
instead it prints a ✅ or ❌ per check and keeps going, so you can see
everything that's still broken in one run.

## Setup

Run this once. `nltk` is only needed for the optional "try it on a real
corpus" cells later — the required exercises and their checks use small
corpora defined directly in this notebook, so you can complete (and
check) every required exercise with **zero external downloads**.

In [ ]:
# Do not modify

import math
import random
import re
import json
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

random.seed(0)  # reproducibility for any cell that samples

# ---------------------------------------------------------------------
# Checks-cell infrastructure.
# ---------------------------------------------------------------------
_check_results = []


def check(fn, label):
    # Run fn() (a zero-arg callable) and report pass/fail without
    # stopping the cell, even if fn() raises (e.g. because you haven't
    # implemented something yet).
    try:
        ok = bool(fn())
    except Exception as e:
        print(f"❌ {label} — raised {type(e).__name__}: {e}")
        _check_results.append(False)
        return
    print(f"{'✅' if ok else '❌'} {label}")
    _check_results.append(ok)


def run(fn, label="setup"):
    # Run fn() for its side effect / return value; report a warning
    # (without stopping the cell) if it raises.
    try:
        return fn()
    except Exception as e:
        print(f"⚠️  {label} raised {type(e).__name__}: {e} "
              f"(later checks in this cell will likely fail too)")
        return None


def close(a, b, tol=1e-6):
    # Float-equality helper (like pytest.approx) for use inside check().
    return a is not None and b is not None and abs(a - b) <= tol


def summary():
    n_pass = sum(_check_results)
    n_total = len(_check_results)
    print(f"\n{n_pass}/{n_total} checks passed this run.")
    _check_results.clear()

---
# Part A — N-gram Language Models (5 Marks)

Implement an n-gram language model from scratch: MLE estimation, add-k
smoothing, perplexity, and Shannon-style sampling.

**Do not** use `nltk.lm`, `kenlm`, `srilm`, or any library that estimates
n-gram probabilities for you. You **may** use `nltk` purely for
tokenisation/corpus access later, and Python's `collections` module.

In [ ]:
# Do not modify

START = "<s>"
STOP = "</s>"

# The toy corpus from the Week 2 slides -- use it to hand-check your
# bigram probabilities against the lecture's own worked numbers.
SAM_CORPUS = [
    "I am Sam".split(),
    "Sam I am".split(),
    "I do not like green eggs and ham".split(),
]

### `pad_sentence(tokens, n)`
Pad a tokenised sentence with `(n-1)` START tokens and one STOP token.
For `n=1` (unigram), no START padding is needed (only STOP is appended).

```
pad_sentence(["i", "am", "sam"], 2) == ['<s>', 'i', 'am', 'sam', '</s>']
pad_sentence(["i", "am", "sam"], 3) == ['<s>', '<s>', 'i', 'am', 'sam', '</s>']
```

### `ngrams(tokens, n)`
Return the list of n-grams (as tuples) in an already-padded token list.

In [ ]:
def pad_sentence(tokens, n):
    return [START]*(n - 1) + list(tokens) + [STOP]


def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

In [ ]:
# Do not modify
# Unit tests to to ensure that your two functions work correctly.

check(lambda: pad_sentence(["I", "am", "Sam"], 2) == [START, "I", "am", "Sam", STOP],
      "pad_sentence: bigram padding")
check(lambda: pad_sentence(["I", "am", "Sam"], 3) == [START, START, "I", "am", "Sam", STOP],
      "pad_sentence: trigram padding")
check(lambda: pad_sentence(["I", "am", "Sam"], 1) == ["I", "am", "Sam", STOP],
      "pad_sentence: unigram has no START padding")
check(lambda: ngrams(pad_sentence(["I", "am", "Sam"], 2), 2) ==
      [(START, "I"), ("I", "am"), ("am", "Sam"), ("Sam", STOP)],
      "ngrams: bigram extraction")
summary()

### `NGramLM`
An n-gram language model with add-k smoothing.

- `train(sentences)`: estimate counts from a corpus of tokenised sentences.
  Populate `self.vocab` (every *non-padding* token seen — START/STOP must
  **not** be added, since that would corrupt the `|V|` term used in
  smoothing), `self.ngram_counts`, and `self.context_counts`.

- `prob(word, context)`: smoothed conditional probability, using add-k smoothing:

  $$P(word \mid context) = \frac{count(context, word) + k}{count(context) + k \cdot (|V| + 1)}$$

  The **`+1`** matters: STOP is a valid outcome word that is deliberately
  excluded from `self.vocab`, but must still be counted in the
  normalising constant since it *is* a possible value for `word`. Get
  this wrong and `P(. | context)` summed over the vocabulary + STOP will
  not equal 1 (there's a check for exactly this below).

- `sentence_logprob(sentence)`: `log2 P(sentence)` via the chain rule +
  Markov assumption. Sum log2 probabilities term-by-term — do **not**
  multiply raw probabilities — to avoid floating-point underflow on
  longer sentences.

- `perplexity(sentences)`: $PP(W) = 2^{-\frac{1}{N}\sum_i \log_2 P(w_i \mid context_i)}$,
  where $N$ is the total number of scored tokens (including each
  sentence's STOP, excluding the START padding).

- `sample(max_len, seed)`: generate a sentence via Shannon's method —
  repeatedly sample the next word from `P(w | context)` starting from
  `(n-1)` START tokens, until STOP or `max_len` tokens. If `seed` is not
  `None`, seed `random` first (the checks rely on this for reproducibility).

In [ ]:
class NGramLM:
    def __init__(self, n=2, k=1.0):
        if n < 1:
            raise ValueError("n must be at least 1")
        if k < 0:
            raise ValueError("k must be non-negative")
        self.n = n
        self.k = k
        self.vocab = set()
        self.context_counts = Counter()
        self.ngram_counts = Counter()

    def train(self, sentences):
        self.vocab = set()
        self.context_counts = Counter()
        self.ngram_counts = Counter()

        for sentence in sentences:
            tokens = list(sentence)
            self.vocab.update(tokens)

            for gram in ngrams(pad_sentence(tokens, self.n), self.n):
                context = gram[:-1]
                self.ngram_counts[gram] += 1
                self.context_counts[context] += 1

    def prob(self, word, context):
        context = tuple(context) if self.n > 1 else ()
        gram = context + (word,)
        denominator = self.context_counts[context] + self.k * (len(self.vocab) + 1)

        if denominator == 0:
            return 0.0
        return (self.ngram_counts[gram] + self.k) / denominator

    def sentence_logprob(self, sentence):
        log_probability = 0.0
        for gram in ngrams(pad_sentence(list(sentence), self.n), self.n):
            probability = self.prob(gram[-1], gram[:-1])
            if probability == 0.0:
                return float("-inf")
            log_probability += math.log2(probability)
        return log_probability

    def perplexity(self, sentences):
        total_logprob = 0.0
        token_count = 0

        for sentence in sentences:
            tokens = list(sentence)
            total_logprob += self.sentence_logprob(tokens)
            token_count += len(tokens) + 1

        if token_count == 0:
            raise ValueError("perplexity requires at least one sentence")
        return 2 ** (-total_logprob / token_count)

    def sample(self, max_len=20, seed=None):
        if seed is not None:
            random.seed(seed)

        output = []
        context = (START,) * (self.n - 1)
        candidates = sorted(self.vocab | {STOP})

        for _ in range(max_len):
            weights = [self.prob(word, context) for word in candidates]
            word = random.choices(candidates, weights=weights, k=1)[0]
            output.append(word)

            if word == STOP:
                break
            context = () if self.n == 1 else (context + (word,))[-(self.n - 1):]

        return output

In [ ]:
# Do not modify
# Unit tests for your new class

lm = run(lambda: NGramLM(n=2, k=0.0), "NGramLM(k=0.0)")
run(lambda: lm.train(SAM_CORPUS), "lm.train(SAM_CORPUS)")

# These numbers come straight from the Week 2 'Example' slide.
check(lambda: close(lm.prob("I", (START,)), 2 / 3), "P(I|<s>) == 0.67 (no smoothing)")
check(lambda: close(lm.prob("Sam", (START,)), 1 / 3), "P(Sam|<s>) == 0.33")
check(lambda: close(lm.prob("am", ("I",)), 2 / 3), "P(am|I) == 0.67")
check(lambda: close(lm.prob(STOP, ("Sam",)), 1 / 2), "P(</s>|Sam) == 0.5")
check(lambda: close(lm.prob("Sam", ("am",)), 1 / 2), "P(Sam|am) == 0.5")
check(lambda: lm.prob("I", ("ham",)) == 0.0, "unseen bigram has zero prob WITHOUT smoothing")

lm_smooth = run(lambda: NGramLM(n=2, k=1.0), "NGramLM(k=1.0)")
run(lambda: lm_smooth.train(SAM_CORPUS), "lm_smooth.train(SAM_CORPUS)")
check(lambda: lm_smooth.prob("I", ("ham",)) > 0.0, "add-1 smoothing removes zero probability")
check(lambda: close(sum(lm_smooth.prob(w, ("I",)) for w in lm_smooth.vocab | {STOP}), 1.0),
      "P(.|context) sums to 1 over vocab + STOP (the |V|+1 normaliser)")
check(lambda: lm_smooth.perplexity([["I", "am", "Sam"]]) <
      lm_smooth.perplexity([["Sam", "Sam", "Sam", "Sam"]]),
      "perplexity prefers a training-like sentence over a weird one")
check(lambda: math.isfinite(lm_smooth.perplexity([["I", "like", "ham"]])) and
      lm_smooth.perplexity([["I", "like", "ham"]]) >= 1.0,
      "perplexity is finite and >= 1 with smoothing")

uni = run(lambda: NGramLM(n=1, k=1.0), "NGramLM(n=1)")
run(lambda: uni.train(SAM_CORPUS), "uni.train")
tri = run(lambda: NGramLM(n=3, k=1.0), "NGramLM(n=3)")
run(lambda: tri.train(SAM_CORPUS), "tri.train")
check(lambda: tri.perplexity(SAM_CORPUS) <= uni.perplexity(SAM_CORPUS),
      "trigram training perplexity <= unigram training perplexity")

sample_out = run(lambda: lm_smooth.sample(max_len=20, seed=42), "lm_smooth.sample")
check(lambda: isinstance(sample_out, list) and len(sample_out) > 0 and
      sample_out[-1] == STOP and START not in sample_out,
      "sample() returns a list ending in STOP, no leading START")
summary()

### Now try it on a real corpus.

Train unigram/bigram/trigram models on a real corpus — `nltk.corpus.brown`
is a good default (remember to hold out a test set!) — and report
perplexity for each order, with and without smoothing.


In [ ]:
# Do not modify. This should do a full test of your implementation using a real dataset.
try:
    import nltk
    nltk.data.find("corpora/brown")
    from nltk.corpus import brown

    sents = [[w.lower() for w in s] for s in brown.sents()[:2000]]
    random.Random(0).shuffle(sents)
    split = int(0.9 * len(sents))
    train_sents, test_sents = sents[:split], sents[split:]

    for order in (1, 2, 3):
        m = NGramLM(n=order, k=1.0)
        m.train(train_sents)
        print(f"n={order}: test perplexity = {m.perplexity(test_sents):.2f}")
except LookupError:
    print("Brown corpus not downloaded -- run:\n"
          "  import nltk; nltk.download('brown')\n"
          "then re-run this cell.")
except NotImplementedError:
    print("Skipping -- looks like Part A isn't fully implemented yet. "
          "Come back to this cell once the checks above are all green.")

---
# Part B — Part-of-Speech Tagging with a Hidden Markov Model (5 Marks)

Implement an HMM POS tagger from scratch: parameter estimation
(transition matrix A, emission matrix B, initial distribution π) via MLE
+ add-k smoothing, and the **Viterbi algorithm** for decoding.

**Do not** use `nltk.tag.hmm`, `sklearn-crfsuite`, spaCy's tagger, or any
pretrained/off-the-shelf tagger.

### Tagset gotcha (read before you train on a real corpus)

nltk's built-in `"universal"` tagset (`tagged_sents(tagset="universal")`)
merges proper nouns into `NOUN` — it has **no separate `PROPN` tag**.
Since Part D's person/location extraction looks for `PROPN`, training on
that tagset as-is would silently break Part D later. The cell below
provides a small Penn Treebank → coarse tagset mapping (with `PROPN`)
that matches the toy tagset used throughout this notebook. It's given
complete — you don't need to (and shouldn't) modify it, just use it when
you get to the "real corpus" cell.

In [ ]:
# Do not modify
# Provided complete -- Penn Treebank tag -> coarse tag used throughout
# this notebook. See the Week 3 "Penn Treebank Tagset" slide for what
# each Penn tag means.
PENN_TO_COARSE = {
    "NN": "NOUN", "NNS": "NOUN",
    "NNP": "PROPN", "NNPS": "PROPN",
    "PRP": "PRON", "PRP$": "PRON", "WP": "PRON", "WP$": "PRON", "EX": "PRON",
    "DT": "DET", "PDT": "DET", "WDT": "DET",
    "JJ": "ADJ", "JJR": "ADJ", "JJS": "ADJ",
    "RB": "ADV", "RBR": "ADV", "RBS": "ADV", "WRB": "ADV",
    "VB": "VERB", "VBD": "VERB", "VBG": "VERB", "VBN": "VERB",
    "VBP": "VERB", "VBZ": "VERB",
    "MD": "AUX",
    "IN": "ADP", "TO": "ADP",
    "CC": "CCONJ",
    "CD": "NUM",
    "RP": "PART", "POS": "PART", "UH": "INTJ", "FW": "X", "SYM": "SYM", "LS": "X",
    ".": "PUNCT", ",": "PUNCT", ":": "PUNCT", "``": "PUNCT", "''": "PUNCT",
    "-LRB-": "PUNCT", "-RRB-": "PUNCT", "$": "SYM", "#": "SYM",
}


def coarsen_tag(penn_tag):
    return PENN_TO_COARSE.get(penn_tag, penn_tag)


def coarsen_sentence(tagged_sentence):
    return [(word, coarsen_tag(tag)) for word, tag in tagged_sentence]


START_TAG = "<START>"

# A small, hand-constructed toy corpus (engineered so the correct
# Viterbi output is checkable by hand) recreating the "We can fish"
# ambiguity from lecture.
POS_TRAIN_CORPUS = [
    [("We", "PRON"), ("can", "AUX"), ("fish", "VERB")],
    [("We", "PRON"), ("will", "AUX"), ("go", "VERB")],
    [("They", "PRON"), ("can", "AUX"), ("see", "VERB")],
    [("I", "PRON"), ("can", "AUX"), ("swim", "VERB")],
    [("The", "DET"), ("can", "NOUN"), ("is", "AUX"), ("empty", "ADJ")],
    [("Open", "VERB"), ("the", "DET"), ("can", "NOUN")],
    [("Janet", "NOUN"), ("will", "AUX"), ("back", "VERB"), ("the", "DET"), ("bill", "NOUN")],
    [("The", "DET"), ("committee", "NOUN"), ("will", "AUX"), ("back", "VERB"),
     ("the", "DET"), ("bill", "NOUN")],
]

### `HMMTagger`

- `train(tagged_sentences)`: populate `self.tags`, `self.vocab`,
  `self.tag_counts`, `self.transition_counts[(prev_tag, tag)]`,
  `self.initial_counts[tag]` (count of times `tag` is the *first* tag of
  a sentence), `self.emission_counts[(tag, word)]`, `self.num_sentences`.

- `transition_prob(prev_tag, tag)`: $P(tag \mid prev\_tag)$ with add-k
  smoothing over `self.tags`. If `prev_tag == START_TAG`, this is the
  *initial* distribution — use `self.initial_counts`/`self.num_sentences`
  instead of `self.transition_counts`/`self.tag_counts`.

  $$P(tag \mid prev\_tag) = \frac{count(prev\_tag, tag) + k}{count(prev\_tag) + k \cdot |tags|}$$

- `emission_prob(tag, word)`: $P(word \mid tag)$ with add-k smoothing
  over `(|vocab| + 1)` — the `+1` reserves probability mass for an
  implicit `<UNK>` so unseen words still get a nonzero emission under
  every tag.

- `viterbi(sentence)`: the most probable tag sequence, computed **in
  log-space** (sum of log-probabilities) to avoid underflow — exactly as
  for language models, and equally necessary here.

- `evaluate(test_sentences)`: per-token tagging accuracy.

### `most_frequent_tag_baseline(train_sentences)`
Map each observed word to its single most frequent training tag. This is
the baseline your HMM should beat — if it doesn't on held-out data,
something in `transition_prob`/`emission_prob`/`viterbi` is likely wrong.

In [ ]:
class HMMTagger:
    def __init__(self, k=1.0):
        self.k = k
        self.tags = []
        self.vocab = set()
        self.transition_counts = Counter()
        self.tag_counts = Counter()
        self.emission_counts = Counter()
        self.initial_counts = Counter()
        self.num_sentences = 0
        self.transition_context_counts = Counter()

    def train(self, tagged_sentences):
        self.tags = []
        self.vocab = set()
        self.transition_counts = Counter()
        self.tag_counts = Counter()
        self.emission_counts = Counter()
        self.initial_counts = Counter()
        self.num_sentences = 0
        self.transition_context_counts = Counter()

        tag_set = set()
        for sentence in tagged_sentences:
            tagged_tokens = list(sentence)
            if not tagged_tokens:
                continue

            self.num_sentences += 1
            self.initial_counts[tagged_tokens[0][1]] += 1

            for word, tag in tagged_tokens:
                self.vocab.add(word)
                tag_set.add(tag)
                self.tag_counts[tag] += 1
                self.emission_counts[(tag, word)] += 1

            for (_, prev_tag), (_, tag) in zip(tagged_tokens, tagged_tokens[1:]):
                self.transition_counts[(prev_tag, tag)] += 1
                self.transition_context_counts[prev_tag] += 1

        self.tags = sorted(tag_set)

    def transition_prob(self, prev_tag, tag):
        if tag not in self.tags:
            return 0.0

        if prev_tag == START_TAG:
            count = self.initial_counts[tag]
            total = self.num_sentences
        else:
            count = self.transition_counts[(prev_tag, tag)]
            total = self.transition_context_counts[prev_tag]

        denominator = total + self.k * len(self.tags)
        if denominator == 0:
            return 0.0
        return (count + self.k) / denominator

    def emission_prob(self, tag, word):
        if tag not in self.tags:
            return 0.0

        denominator = self.tag_counts[tag] + self.k * (len(self.vocab) + 1)
        if denominator == 0:
            return 0.0
        return (self.emission_counts[(tag, word)] + self.k) / denominator

    def viterbi(self, sentence):
        words = list(sentence)
        if not words or not self.tags:
            return []

        def log_prob(probability):
            return math.log2(probability) if probability > 0 else float("-inf")

        scores = {}
        backpointers = []
        for tag in self.tags:
            scores[tag] = (
                log_prob(self.transition_prob(START_TAG, tag))
                + log_prob(self.emission_prob(tag, words[0]))
            )
        backpointers.append({})

        for word in words[1:]:
            next_scores = {}
            pointers = {}
            for tag in self.tags:
                best_prev = max(
                    self.tags,
                    key=lambda prev_tag: scores[prev_tag]
                    + log_prob(self.transition_prob(prev_tag, tag)),
                )
                next_scores[tag] = (
                    scores[best_prev]
                    + log_prob(self.transition_prob(best_prev, tag))
                    + log_prob(self.emission_prob(tag, word))
                )
                pointers[tag] = best_prev
            scores = next_scores
            backpointers.append(pointers)

        best_tag = max(self.tags, key=lambda tag: scores[tag])
        best_path = [best_tag]
        for position in range(len(words) - 1, 0, -1):
            best_tag = backpointers[position][best_tag]
            best_path.append(best_tag)
        return list(reversed(best_path))

    def evaluate(self, test_sentences):
        correct = total = 0
        for sentence in test_sentences:
            words = [word for word, _ in sentence]
            gold_tags = [tag for _, tag in sentence]
            predicted_tags = self.viterbi(words)
            correct += sum(predicted == gold for predicted, gold in zip(predicted_tags, gold_tags))
            total += len(gold_tags)
        return correct / total if total else 0.0


def most_frequent_tag_baseline(train_sentences):
    word_tag_counts = {}
    for sentence in train_sentences:
        for word, tag in sentence:
            word_tag_counts.setdefault(word, Counter())[tag] += 1
    return {word: counts.most_common(1)[0][0] for word, counts in word_tag_counts.items()}

In [ ]:
# Do not modify
# Unit tests for your new HMM Tagger class

tagger = run(lambda: HMMTagger(k=1.0), "HMMTagger(k=1.0)")
run(lambda: tagger.train(POS_TRAIN_CORPUS), "tagger.train")

check(lambda: set(tagger.tags) == {"PRON", "AUX", "VERB", "DET", "NOUN", "ADJ"},
      "train() populates self.tags correctly")
check(lambda: "can" in tagger.vocab and "fish" in tagger.vocab,
      "train() populates self.vocab correctly")
check(lambda: close(sum(tagger.transition_prob("AUX", t) for t in tagger.tags), 1.0),
      "transition_prob sums to 1 over self.tags")
check(lambda: close(sum(tagger.emission_prob("NOUN", w) for w in tagger.vocab) +
      tagger.emission_prob("NOUN", "<UNK>"), 1.0),
      "emission_prob sums to 1 over vocab + <UNK>")
check(lambda: (tagger.transition_prob("PRON", "AUX") * tagger.emission_prob("AUX", "can")) >
      (tagger.transition_prob("PRON", "NOUN") * tagger.emission_prob("NOUN", "can")),
      "'can' after a PRON scores higher as AUX than as NOUN")
check(lambda: tagger.viterbi(["We", "can", "fish"]) == ["PRON", "AUX", "VERB"],
      "viterbi disambiguates 'We can fish' as PRON AUX VERB")
check(lambda: len(tagger.viterbi(["The", "committee", "will", "back", "the", "bill"])) == 6,
      "viterbi output length matches input length")
check(lambda: len(tagger.viterbi(["We", "can", "juggle"])) == 3,
      "viterbi handles an out-of-vocabulary word without crashing")

light = run(lambda: HMMTagger(k=0.01), "HMMTagger(k=0.01)")
run(lambda: light.train(POS_TRAIN_CORPUS), "light.train")
check(lambda: close(light.evaluate(POS_TRAIN_CORPUS), 1.0),
      "a lightly-smoothed tagger recovers this tiny corpus perfectly "
      "(k=1.0 over-smooths a corpus this small -- see the note below)")

baseline = run(lambda: most_frequent_tag_baseline(POS_TRAIN_CORPUS), "most_frequent_tag_baseline")
check(lambda: baseline["can"] == "AUX", "baseline: 'can' -> AUX (most frequent in training)")
check(lambda: tagger.viterbi(["Open", "the", "can"])[-1] == "NOUN" and baseline["can"] != "NOUN",
      "HMM uses context to get 'Open the can' right where the context-free baseline can't")
summary()

> **Why does `k=1.0` need a lighter smoothing constant (`k=0.01`) to hit
> 100% on its own training data above?** On a corpus this tiny, `k=1.0`
> smooths away enough of the (often single-instance) signal for words
> like "Janet" or "empty" that Viterbi can pick a different-but-plausible
> tag than the gold one. That's not a bug in your code — it's a genuine,
> discussion-worthy property of add-k smoothing on small data.

### Now try it for real (not auto-checked)

Train on `nltk.corpus.treebank` (raw Penn Treebank tags, **not**
`tagset="universal"` — see the gotcha above) run through
`coarsen_sentence`, and report accuracy against
`most_frequent_tag_baseline` on a held-out split.

In [ ]:
#Do not modify. Run on a real dataset.
try:
    import nltk
    nltk.data.find("corpora/treebank")
    from nltk.corpus import treebank

    sents = [coarsen_sentence(s) for s in treebank.tagged_sents()]
    random.Random(0).shuffle(sents)
    split = int(0.9 * len(sents))
    train_sents, test_sents = sents[:split], sents[split:]

    real_tagger = HMMTagger(k=0.1)
    real_tagger.train(train_sents)
    print(f"HMM accuracy on held-out Treebank test set: {real_tagger.evaluate(test_sents):.3f}")

    real_baseline = most_frequent_tag_baseline(train_sents)
    correct = total = 0
    for sentence in test_sents:
        for word, gold in sentence:
            total += 1
            if real_baseline.get(word, "NOUN") == gold:
                correct += 1
    print(f"Most-frequent-tag baseline accuracy: {correct / total:.3f}")
except LookupError:
    print("Penn Treebank sample not downloaded -- run:\n"
          "  import nltk; nltk.download('treebank')\n"
          "then re-run this cell.")
except NotImplementedError:
    print("Skipping -- looks like Part B isn't fully implemented yet. "
          "Come back to this cell once the checks above are all green.")

---
# Part C — End-to-End Fact  (5 Mark)

**Part C1**: a minimal rule-based Named Entity Recogniser (gazetteer +
regex + POS heuristics) layered on your Part B tagger, identifying
PERSON, LOCATION, and DATE spans. Full statistical/neural NER is out of
scope this early — it returns later in the course.

**Part C2**: wire Parts B and C1 into a single pipeline mapping a raw
sentence to a structured fact.

**Do not** use spaCy's NER, `nltk.chunk.ne_chunk`, or any pretrained NER
model.

The constants below (`MONTHS`, `LOCATION_GAZETTEER`, `DATE_PATTERN`,
`LOCATION_PREPOSITIONS`) are provided. **Note: `LOCATION_GAZETTEER`
deliberately does not cover every location you'll see later** —
extending it, and/or leaning more on your tagger's PROPN predictions than
the gazetteer, is part of the exercise (and a good discussion point about
why gazetteer-based NER is brittle).

In [ ]:
# Do not modify
MONTHS = {
    "january", "february", "march", "april", "may", "june",
    "july", "august", "september", "october", "november", "december",
}

LOCATION_GAZETTEER = {
    "ulm", "germany", "england", "london", "paris", "france",
    "cambridge", "oxford", "vienna", "austria", "warsaw", "poland",
    "italy", "pisa", "scotland", "edinburgh",
}

#A truly filthy regex to cover every type of date format you will encounter.
DATE_PATTERN = re.compile(
    r"\b\d{1,2}\s+(?:" + "|".join(MONTHS) + r")\s+\d{3,4}\b",
    re.IGNORECASE,
)

LOCATION_PREPOSITIONS = {"in", "at", "from", "near"}

### `find_dates(text)`
Return all DATE spans in `text` matching `DATE_PATTERN` (e.g. "14 March
1879"), preserving the original casing/text of each match.

### `find_locations(tagged_sentence)`
Return LOCATION spans: maximal runs of consecutive `PROPN` tokens
immediately following a preposition in `LOCATION_PREPOSITIONS`, **or**
any single token whose lower-cased form is in `LOCATION_GAZETTEER`
(regardless of its POS tag, to hedge against tagger mistakes). Left-to-right
order, no duplicates.

### `find_person(tagged_sentence)`
Return the text of the **first** maximal run of consecutive `PROPN`
tokens (typically the subject in our biography sentences), or `None`.

In [ ]:
def find_dates(text):
    return [match.group(0) for match in DATE_PATTERN.finditer(text)]


def find_locations(tagged_sentence):
    locations = []
    seen = set()

    def add_location(location):
        key = location.lower()
        if key not in seen:
            seen.add(key)
            locations.append(location)

    index = 0
    while index < len(tagged_sentence):
        word, tag = tagged_sentence[index]

        if word.lower() in LOCATION_PREPOSITIONS:
            end = index + 1
            while end < len(tagged_sentence) and tagged_sentence[end][1] == "PROPN":
                end += 1
            if end > index + 1:
                add_location(" ".join(token for token, _ in tagged_sentence[index + 1:end]))
                index = end
                continue

        if word.lower() in LOCATION_GAZETTEER:
            add_location(word)
        index += 1

    return locations


def find_person(tagged_sentence):
    index = 0
    while index < len(tagged_sentence):
        if tagged_sentence[index][1] != "PROPN":
            index += 1
            continue

        end = index
        while end < len(tagged_sentence) and tagged_sentence[end][1] == "PROPN":
            end += 1
        return " ".join(word for word, _ in tagged_sentence[index:end])

    return None

In [ ]:
#Do not modify
#Unit tests for entity extraction
check(lambda: find_dates("Albert Einstein was born in Ulm on 14 March 1879") ==
      ["14 March 1879"], "find_dates: single match")
check(lambda: find_dates("Marie Curie won the Nobel Prize in 1903") == [],
      "find_dates: no match on a year-only date")
check(lambda: find_dates("Born on 1 January 1900, died on 2 February 2000") ==
      ["1 January 1900", "2 February 2000"], "find_dates: multiple matches")

tagged = [("Albert", "PROPN"), ("Einstein", "PROPN"), ("was", "AUX"),
          ("born", "VERB"), ("in", "ADP"), ("Ulm", "PROPN")]
check(lambda: find_locations(tagged) == ["Ulm"], "find_locations: preposition + PROPN")
check(lambda: find_locations([("She", "PRON"), ("came", "VERB"), ("from", "ADP"),
      ("New", "PROPN"), ("York", "PROPN")]) == ["New York"],
      "find_locations: multi-word span")
check(lambda: find_locations([("born", "VERB"), ("in", "ADP"), ("Ulm", "NOUN")]) == ["Ulm"],
      "find_locations: gazetteer fallback even if tagger mistagged PROPN as NOUN")
check(lambda: find_locations([("in", "ADP"), ("Ulm", "PROPN"), ("or", "CCONJ"),
      ("in", "ADP"), ("Ulm", "PROPN")]) == ["Ulm"], "find_locations: no duplicates")

check(lambda: find_person(tagged) == "Albert Einstein", "find_person: first PROPN run")
check(lambda: find_person([("The", "DET"), ("dog", "NOUN"), ("barked", "VERB")]) is None,
      "find_person: None when there's no PROPN")
summary()

### `extract_facts(sentence, tagger)` and `evaluate_pipeline(gold, tagger)`

`extract_facts` returns a dict with keys `"who"`, `"did_what"`,
`"where"`, `"when"`:

- `who`: output of `find_person`
- `did_what`: text of the first maximal run of consecutive `AUX`/`VERB`
  tags (e.g. "was born"), joined with spaces, or `None`
- `where`: output of `find_locations`
- `when`: the first date found by `find_dates` on the **original**
  `sentence` string (not the tokenised version), or `None`

`evaluate_pipeline(gold, tagger)` scores `extract_facts` predictions
against a list of gold dicts (`"sentence"`, `"who"`, `"did_what"`,
`"where"` (a list), `"when"`). Score `who`/`did_what`/`when` as exact
match after `.strip().lower()`. Score `where` as **micro-averaged**
precision/recall/F1 over sets of lower-cased strings (sum true
positives/predicted/gold across the whole dataset first, then divide —
not an average of per-example F1s). Return exactly:
`{"who_accuracy", "did_what_accuracy", "when_accuracy",
"where_precision", "where_recall", "where_f1"}`.

In [ ]:
def tokenize(sentence):
    return re.findall(r"[A-Za-z]+|\d+", sentence)


def extract_facts(sentence, tagger):
    words = tokenize(sentence)
    tags = tagger.viterbi(words)
    tagged_sentence = list(zip(words, tags))

    did_what = None
    index = 0
    while index < len(tagged_sentence):
        if tagged_sentence[index][1] not in {"AUX", "VERB"}:
            index += 1
            continue

        end = index
        while end < len(tagged_sentence) and tagged_sentence[end][1] in {"AUX", "VERB"}:
            end += 1
        did_what = " ".join(word for word, _ in tagged_sentence[index:end])
        break

    dates = find_dates(sentence)
    return {
        "who": find_person(tagged_sentence),
        "did_what": did_what,
        "where": find_locations(tagged_sentence),
        "when": dates[0] if dates else None,
    }


def evaluate_pipeline(gold, tagger):
    def normalise(value):
        return value.strip().lower() if value is not None else ""

    who_correct = did_what_correct = when_correct = 0
    true_positives = predicted_locations = gold_locations = 0

    for item in gold:
        prediction = extract_facts(item["sentence"], tagger)
        who_correct += normalise(prediction["who"]) == normalise(item["who"])
        did_what_correct += normalise(prediction["did_what"]) == normalise(item["did_what"])
        when_correct += normalise(prediction["when"]) == normalise(item["when"])

        predicted_where = {normalise(value) for value in prediction["where"] if normalise(value)}
        expected_where = {normalise(value) for value in item["where"] if normalise(value)}
        true_positives += len(predicted_where & expected_where)
        predicted_locations += len(predicted_where)
        gold_locations += len(expected_where)

    total = len(gold)
    precision = true_positives / predicted_locations if predicted_locations else 0.0
    recall = true_positives / gold_locations if gold_locations else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return {
        "who_accuracy": who_correct / total if total else 0.0,
        "did_what_accuracy": did_what_correct / total if total else 0.0,
        "when_accuracy": when_correct / total if total else 0.0,
        "where_precision": precision,
        "where_recall": recall,
        "where_f1": f1,
    }

In [ ]:
# Do not modify
# Unit tests for fact extraction.
PIPELINE_TRAIN_CORPUS = [
    [("Albert", "PROPN"), ("Einstein", "PROPN"), ("was", "AUX"), ("born", "VERB"),
     ("in", "ADP"), ("Ulm", "PROPN"), ("on", "ADP"), ("14", "NUM"),
     ("March", "PROPN"), ("1879", "NUM")],
    [("Marie", "PROPN"), ("Curie", "PROPN"), ("was", "AUX"), ("born", "VERB"),
     ("in", "ADP"), ("Warsaw", "PROPN"), ("on", "ADP"), ("7", "NUM"),
     ("November", "PROPN"), ("1867", "NUM")],
    [("Isaac", "PROPN"), ("Newton", "PROPN"), ("was", "AUX"), ("born", "VERB"),
     ("in", "ADP"), ("England", "PROPN"), ("on", "ADP"), ("25", "NUM"),
     ("December", "PROPN"), ("1642", "NUM")],
]
pipeline_tagger = run(lambda: HMMTagger(k=0.01), "HMMTagger for pipeline")
run(lambda: pipeline_tagger.train(PIPELINE_TRAIN_CORPUS), "pipeline_tagger.train")

facts = run(lambda: extract_facts("Albert Einstein was born in Ulm on 14 March 1879",
                                   pipeline_tagger), "extract_facts(Einstein sentence)")
check(lambda: facts["who"] == "Albert Einstein", "extract_facts: who")
check(lambda: facts["did_what"] == "was born", "extract_facts: did_what")
check(lambda: "Ulm" in facts["where"], "extract_facts: where")
check(lambda: facts["when"] == "14 March 1879", "extract_facts: when")

facts2 = run(lambda: extract_facts("Isaac Newton was born in England", pipeline_tagger),
             "extract_facts(no date in sentence)")
check(lambda: facts2["who"] == "Isaac Newton", "extract_facts: who (2nd example)")
check(lambda: facts2["when"] is None, "extract_facts: when is None when no date present")

perfect_gold = [{
    "sentence": "Albert Einstein was born in Ulm on 14 March 1879",
    "who": "Albert Einstein", "did_what": "was born",
    "where": ["Ulm"], "when": "14 March 1879",
}]
scores = run(lambda: evaluate_pipeline(perfect_gold, pipeline_tagger), "evaluate_pipeline")
check(lambda: set(scores.keys()) == {
      "who_accuracy", "did_what_accuracy", "when_accuracy",
      "where_precision", "where_recall", "where_f1"},
      "evaluate_pipeline returns exactly the expected keys")
check(lambda: close(scores["who_accuracy"], 1.0), "evaluate_pipeline: who_accuracy == 1.0")
check(lambda: close(scores["where_f1"], 1.0), "evaluate_pipeline: where_f1 == 1.0")
summary()

---
## Final demo: the full pipeline on the gold evaluation set

This loads `data/toy_tagged_corpus.txt` (a slightly larger toy corpus)
and `data/biography_sentences.json` (20 hand-annotated biography
sentences), trains a tagger, and evaluates the whole Part D pipeline.

**Calibrate your expectations before you start debugging.** A *correct*
reference implementation, trained on the **full Penn Treebank** (not this
notebook's tiny toy corpus), scores roughly: `did_what_accuracy` and
`when_accuracy` near 1.0, but `who_accuracy` only around **0.1–0.25**,
and `where_recall` around **0.35**. This is not a bug to chase away —
it's a genuine, well-known weakness of bigram HMMs: Treebank is Wall
Street Journal financial news, and "Albert Einstein" is an
out-of-vocabulary bigram of out-of-vocabulary words relative to that
domain. With no reliable emission signal, Viterbi falls back almost
entirely on transition probabilities, and "PROPN PROPN" is simply not as
dominant a transition in WSJ as, say, "PRON AUX" ("it was...", "he
said..."), so `find_person`/`find_locations` frequently grab the wrong
span or nothing at all. **This domain-mismatch finding — not a high
accuracy number — is the intended result of Part D**, and your report's
error analysis should center on it. One concrete, gradeable way to
improve on it: mix a small hand-tagged set of biography-style sentences
into your training data, oversampling if needed, and report how much
that in-domain "seed data" moves the needle.

In [ ]:
# Do not modify
# The moment of truth. Does everything work?

def load_conll(path):
    sentences, current = [], []
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line:
            if current:
                sentences.append(current)
                current = []
            continue
        word, tag = line.split("\t")
        current.append((word, tag))
    if current:
        sentences.append(current)
    return sentences


demo_tagged = run(lambda: load_conll("data/toy_tagged_corpus.txt"), "load_conll")
demo_tagger = run(lambda: HMMTagger(k=0.1), "HMMTagger for demo")
run(lambda: demo_tagger.train(demo_tagged), "demo_tagger.train")

example = "Albert Einstein was born in Ulm on 14 March 1879"
print(f"sentence: {example}")
print(f"facts:    {run(lambda: extract_facts(example, demo_tagger), 'extract_facts')}")

gold = run(lambda: json.loads(Path("data/biography_sentences.json").read_text()), "load gold set")
if gold:
    demo_scores = run(lambda: evaluate_pipeline(gold, demo_tagger), "evaluate_pipeline(full gold set)")
    print("\nScores on the full 20-example gold set (toy-corpus-trained tagger):")
    if demo_scores:
        for k, v in demo_scores.items():
            print(f"  {k}: {v:.3f}")
    print("\n(Train on the full Penn Treebank -- see the Part B 'try it for real' "
          "cell -- for the more realistic numbers discussed above, and include "
          "both in your report.)")

## Training on full Penn Treebank

In [ ]:
import nltk
nltk.download("treebank")

In [ ]:
gold = json.loads(Path("data/biography_sentences.json").read_text())

treebank_scores = evaluate_pipeline(gold, real_tagger)
for metric, value in treebank_scores.items():
    print(f"{metric}: {value:.3f}")